# PA2 Part 1 Colab GPU Test

Use this notebook from VS Code's Colab/Jupyter workflow to run the Part 1 Triton local test on a Colab GPU.

Recommended runtime: **L4** if available, otherwise **T4** is fine for correctness. Official speed is still determined by the course A10 remote grader.

You do **not** need to sync the whole repo. For Part 1 local testing, the notebook embeds `student_kernel.py`, `student_local_test.py`, `student_submit.py`, and `requirements.txt`.

In [9]:
# 1. Confirm that Colab gave this runtime a GPU.
!nvidia-smi

Sat May  9 18:56:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   52C    P8             17W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
# Optional: clear Colab-side caches before recreating files and submitting.
# This does not clear remote grader caches.
!rm -rf ~/.triton/cache
!rm -rf ~/.cache/torch
!rm -rf __pycache__
!find . -maxdepth 2 -type d -name "__pycache__" -print -exec rm -rf {} +


## Create the Part 1 Files

This notebook embeds the files needed for Part 1 local testing and optional remote submission, so you do not need to sync the whole workspace or manually select files. If you later edit `student_kernel.py`, regenerate this notebook or rerun Codex to update the embedded copy.

In [ ]:
# 2. Create the Part 1 files directly in the Colab runtime.
# This avoids slow full-workspace sync and avoids manual file picker uploads.
from pathlib import Path
import base64

embedded_files = {
    'student_kernel.py': 'S0VSTkVMX0NPTkZJR1MgPSBbCiAgICB7IkJMT0NLX00iOiA2NCwgIkJMT0NLX04iOiA2NCwgIkJMT0NLX0siOiAzMiwgIm51bV93YXJwcyI6IDQsICJudW1fc3RhZ2VzIjogNH0sCl0KCgpAdHJpdG9uLmppdApkZWYgbWF0bXVsX2FkZF9yZWx1X2tlcm5lbF9mcDE2KAogICAgYV9wdHIsCiAgICBiX3B0ciwKICAgIGNfcHRyLAogICAgZF9wdHIsCiAgICBNLAogICAgTiwKICAgIEssCiAgICBzdHJpZGVfYW0sCiAgICBzdHJpZGVfYWssCiAgICBzdHJpZGVfYmssCiAgICBzdHJpZGVfYm4sCiAgICBzdHJpZGVfY20sCiAgICBzdHJpZGVfY24sCiAgICBzdHJpZGVfZG0sCiAgICBzdHJpZGVfZG4sCiAgICBCTE9DS19NOiB0bC5jb25zdGV4cHIsCiAgICBCTE9DS19OOiB0bC5jb25zdGV4cHIsCiAgICBCTE9DS19LOiB0bC5jb25zdGV4cHIsCik6CiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgICMgU3RlcCAxOiBUaWxlOiBBc3NpZ25tZW50CiAgICAjCiAgICAjIEVhY2gga2VybmVsIGluc3RhbmNlIGlzIG1hcHBlZCB0byBhIHRpbGUgaW4gdGhlIG91dHB1dCBtYXRyaXggQy4KICAgICMgQ29tcHV0ZSB0aGUgc3RhcnRpbmcgaW5kaWNlcyAobV9zdGFydCwgbl9zdGFydCkgZm9yIHRoaXMgdGlsZS4KICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgIyBUT0RPOiBDb21wdXRlIHRoZSB0aWxlIGluZGljZXMgdXNpbmcgcHJvZ3JhbV9pZCgwKSBmb3IgTSBhbmQgcHJvZ3JhbV9pZCgxKSBmb3IgTi4KICAgIHBpZF9tID0gdGwucHJvZ3JhbV9pZCgwKQogICAgcGlkX24gPSB0bC5wcm9ncmFtX2lkKDEpCgogICAgb2Zmc19tID0gcGlkX20gKiBCTE9DS19NICsgdGwuYXJhbmdlKDAsIEJMT0NLX00pCiAgICBvZmZzX24gPSBwaWRfbiAqIEJMT0NLX04gKyB0bC5hcmFuZ2UoMCwgQkxPQ0tfTikKICAgIG9mZnNfayA9IHRsLmFyYW5nZSgwLCBCTE9DS19LKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgIyBTdGVwIDI6IFJlZ2lzdGVyIFRpbGluZwogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICAjIFRPRE86IEluaXRpYWxpemUgdGhlIGFjY3VtdWxhdG9yICJhY2MiIHdpdGggemVyb3MgKGR0eXBlOiBmbG9hdDE2IG9yIGZsb2F0MzIpLgogICAgYWNjID0gdGwuemVyb3MoKEJMT0NLX00sIEJMT0NLX04pLCBkdHlwZT10bC5mbG9hdDMyKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgIyBTdGVwIDM6IFNoYXJlZCBNZW1vcnkgVGlsaW5nICYgQ29vcGVyYXRpdmUgRmV0Y2hpbmcuCiAgICAjIENvbXB1dGUgcG9pbnRlcnMgdG8gdGhlIHN1Yi10aWxlcyBvZiBBIGFuZCBCIHRoYXQgYXJlIG5lZWRlZCB0byBjb21wdXRlCiAgICAjIHRoZSBjdXJyZW50IEMgdGlsZS4gVGhlIG9mZnNldHMgaGVyZSBzZXJ2ZSB0byBsb2FkIEJMT0NLX00geCBCTE9DS19LCiAgICAjIGFuZCBCTE9DS19LIHggQkxPQ0tfTiBibG9ja3MgZnJvbSBBIGFuZCBCIHJlc3BlY3RpdmVseS4KICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgIyBUT0RPOiBGaW5pc2ggY29kZSBiZWxvdy4KICAgIGZvciBrIGluIHJhbmdlKDAsIEssIEJMT0NLX0spOgogICAgICAgIGFfcHRycyA9IGFfcHRyICsgKG9mZnNfbVs6LCBOb25lXSAqIHN0cmlkZV9hbSArIChrICsgb2Zmc19rW05vbmUsIDpdKSAqIHN0cmlkZV9haykKICAgICAgICBiX3B0cnMgPSBiX3B0ciArICgoayArIG9mZnNfa1s6LCBOb25lXSkgKiBzdHJpZGVfYmsgKyBvZmZzX25bTm9uZSwgOl0gKiBzdHJpZGVfYm4pCgogICAgICAgIGFfdGlsZSA9IHRsLmxvYWQoCiAgICAgICAgICAgIGFfcHRycywKICAgICAgICAgICAgbWFzaz0ob2Zmc19tWzosIE5vbmVdIDwgTSkgJiAoKGsgKyBvZmZzX2tbTm9uZSwgOl0pIDwgSyksCiAgICAgICAgICAgIG90aGVyPTAuMCwKICAgICAgICApCiAgICAgICAgYl90aWxlID0gdGwubG9hZCgKICAgICAgICAgICAgYl9wdHJzLAogICAgICAgICAgICBtYXNrPSgoayArIG9mZnNfa1s6LCBOb25lXSkgPCBLKSAmIChvZmZzX25bTm9uZSwgOl0gPCBOKSwKICAgICAgICAgICAgb3RoZXI9MC4wLAogICAgICAgICkKICAgICAgICBhY2MgKz0gdGwuZG90KGFfdGlsZSwgYl90aWxlKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgIyBTdGVwIDQ6IEFkZCBDIGFuZCBBcHBseSBSZUxVIHRvIHRoZSBhY2N1bXVsYXRvcgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICAjIFRPRE86IEZpbmlzaCBjb2RlIGJlbG93LgogICAgY19wdHJzID0gY19wdHIgKyAob2Zmc19tWzosIE5vbmVdICogc3RyaWRlX2NtICsgb2Zmc19uW05vbmUsIDpdICogc3RyaWRlX2NuKQogICAgbWFzayA9IChvZmZzX21bOiwgTm9uZV0gPCBNKSAmIChvZmZzX25bTm9uZSwgOl0gPCBOKQogICAgY190aWxlID0gdGwubG9hZChjX3B0cnMsIG1hc2s9bWFzaywgb3RoZXI9MC4wKQogICAgb3V0ID0gdGwubWF4aW11bShhY2MgKyBjX3RpbGUsIDAuMCkKCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgICMgU3RlcCA1OiBXcml0ZSBDYWNoZSAvIEVwaWxvZ3VlIEZ1c2lvbjogV3JpdGUgdGhlIGNvbXB1dGVkIHRpbGUgdG8gRC4KICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgIyBUT0RPOiBGaW5pc2ggY29kZSBiZWxvdy4KICAgIGRfcHRycyA9IGRfcHRyICsgKG9mZnNfbVs6LCBOb25lXSAqIHN0cmlkZV9kbSArIG9mZnNfbltOb25lLCA6XSAqIHN0cmlkZV9kbikKICAgIHRsLnN0b3JlKGRfcHRycywgb3V0LCBtYXNrPW1hc2spCg==',
    'student_local_test.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBhc3QKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwppbXBvcnQgdGltZQppbXBvcnQgdHJhY2ViYWNrCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgdG9yY2gKaW1wb3J0IHRyaXRvbgppbXBvcnQgdHJpdG9uLmxhbmd1YWdlIGFzIHRsCgoKTSA9IDIwNDgKTiA9IDIwNDgKSyA9IDIwNDgKV0FSTVVQX0lURVJTID0gMjAKQkVOQ0hNQVJLX1JFUEVBVFMgPSA1MDAKU0VBUkNIX1dBUk1VUF9JVEVSUyA9IDMKU0VBUkNIX1JFUEVBVFMgPSAxMApDT1JSRUNUTkVTU19TSEFQRSA9IDUxMgpBVE9MID0gMC4xNQpSVE9MID0gMC4wMzIKS0VSTkVMX0ZVTkNUSU9OX05BTUUgPSAibWF0bXVsX2FkZF9yZWx1X2tlcm5lbF9mcDE2IgpDT05GSUdTX05BTUUgPSAiS0VSTkVMX0NPTkZJR1MiCk1JTl9DT05GSUdTID0gMQpNQVhfQ09ORklHUyA9IDUKRVhQRUNURURfS0VSTkVMX0FSR1MgPSBbCiAgICAiYV9wdHIiLAogICAgImJfcHRyIiwKICAgICJjX3B0ciIsCiAgICAiZF9wdHIiLAogICAgIk0iLAogICAgIk4iLAogICAgIksiLAogICAgInN0cmlkZV9hbSIsCiAgICAic3RyaWRlX2FrIiwKICAgICJzdHJpZGVfYmsiLAogICAgInN0cmlkZV9ibiIsCiAgICAic3RyaWRlX2NtIiwKICAgICJzdHJpZGVfY24iLAogICAgInN0cmlkZV9kbSIsCiAgICAic3RyaWRlX2RuIiwKICAgICJCTE9DS19NIiwKICAgICJCTE9DS19OIiwKICAgICJCTE9DS19LIiwKXQpFWFBFQ1RFRF9DT05GSUdfS0VZUyA9IFsKICAgICJCTE9DS19NIiwKICAgICJCTE9DS19OIiwKICAgICJCTE9DS19LIiwKICAgICJudW1fd2FycHMiLAogICAgIm51bV9zdGFnZXMiLApdCgoKZGVmIF9pc190cml0b25faml0X2RlY29yYXRvcihub2RlOiBhc3QuQVNUKSAtPiBib29sOgogICAgcmV0dXJuICgKICAgICAgICBpc2luc3RhbmNlKG5vZGUsIGFzdC5BdHRyaWJ1dGUpCiAgICAgICAgYW5kIG5vZGUuYXR0ciA9PSAiaml0IgogICAgICAgIGFuZCBpc2luc3RhbmNlKG5vZGUudmFsdWUsIGFzdC5OYW1lKQogICAgICAgIGFuZCBub2RlLnZhbHVlLmlkID09ICJ0cml0b24iCiAgICApCgoKZGVmIG5vcm1hbGl6ZV9jb25maWdzKGNvbmZpZ3M6IEFueSkgLT4gdHVwbGVbYm9vbCwgc3RyLCBsaXN0W2RpY3Rbc3RyLCBpbnRdXSB8IE5vbmVdOgogICAgaWYgbm90IGlzaW5zdGFuY2UoY29uZmlncywgbGlzdCk6CiAgICAgICAgcmV0dXJuIEZhbHNlLCBmIntDT05GSUdTX05BTUV9IG11c3QgYmUgYSBsaXN0IG9mIGRpY3Rpb25hcmllcy4iLCBOb25lCiAgICBpZiBub3QgTUlOX0NPTkZJR1MgPD0gbGVuKGNvbmZpZ3MpIDw9IE1BWF9DT05GSUdTOgogICAgICAgIHJldHVybiAoCiAgICAgICAgICAgIEZhbHNlLAogICAgICAgICAgICBmIntDT05GSUdTX05BTUV9IG11c3QgY29udGFpbiBiZXR3ZWVuIHtNSU5fQ09ORklHU30gYW5kIHtNQVhfQ09ORklHU30gY29uZmlncy4iLAogICAgICAgICAgICBOb25lLAogICAgICAgICkKCiAgICBub3JtYWxpemVkOiBsaXN0W2RpY3Rbc3RyLCBpbnRdXSA9IFtdCiAgICBleHBlY3RlZF9rZXlzID0gc2V0KEVYUEVDVEVEX0NPTkZJR19LRVlTKQogICAgZm9yIGlkeCwgY29uZmlnIGluIGVudW1lcmF0ZShjb25maWdzKToKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShjb25maWcsIGRpY3QpOgogICAgICAgICAgICByZXR1cm4gRmFsc2UsIGYie0NPTkZJR1NfTkFNRX1be2lkeH1dIG11c3QgYmUgYSBkaWN0aW9uYXJ5LiIsIE5vbmUKICAgICAgICBpZiBzZXQoY29uZmlnKSAhPSBleHBlY3RlZF9rZXlzOgogICAgICAgICAgICByZXR1cm4gKAogICAgICAgICAgICAgICAgRmFsc2UsCiAgICAgICAgICAgICAgICBmIntDT05GSUdTX05BTUV9W3tpZHh9XSBtdXN0IGNvbnRhaW4gZXhhY3RseSB0aGVzZSBrZXlzOiAiCiAgICAgICAgICAgICAgICBmInsnLCAnLmpvaW4oRVhQRUNURURfQ09ORklHX0tFWVMpfSIsCiAgICAgICAgICAgICAgICBOb25lLAogICAgICAgICAgICApCgogICAgICAgIG5vcm1hbGl6ZWRfY29uZmlnOiBkaWN0W3N0ciwgaW50XSA9IHt9CiAgICAgICAgZm9yIGtleSBpbiBFWFBFQ1RFRF9DT05GSUdfS0VZUzoKICAgICAgICAgICAgdmFsdWUgPSBjb25maWdba2V5XQogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgaW50KSBvciBpc2luc3RhbmNlKHZhbHVlLCBib29sKSBvciB2YWx1ZSA8PSAwOgogICAgICAgICAgICAgICAgcmV0dXJuIEZhbHNlLCBmIntDT05GSUdTX05BTUV9W3tpZHh9XVsne2tleX0nXSBtdXN0IGJlIGEgcG9zaXRpdmUgaW50ZWdlci4iLCBOb25lCiAgICAgICAgICAgIG5vcm1hbGl6ZWRfY29uZmlnW2tleV0gPSBpbnQodmFsdWUpCiAgICAgICAgbm9ybWFsaXplZC5hcHBlbmQobm9ybWFsaXplZF9jb25maWcpCgogICAgcmV0dXJuIFRydWUsICJvayIsIG5vcm1hbGl6ZWQKCgpkZWYgdmFsaWRhdGVfc291cmNlKHBhdGg6IFBhdGgpIC0+IE5vbmU6CiAgICBzb3VyY2UgPSBwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgdHJlZSA9IGFzdC5wYXJzZShzb3VyY2UpCgogICAgaWYgbGVuKHRyZWUuYm9keSkgIT0gMjoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgIGYiU3VibWlzc2lvbiBtdXN0IGNvbnRhaW4gZXhhY3RseSB0d28gdG9wLWxldmVsIGRlZmluaXRpb25zOiB7Q09ORklHU19OQU1FfSBhbmQgIgogICAgICAgICAgICBmIntLRVJORUxfRlVOQ1RJT05fTkFNRX0uIgogICAgICAgICkKCiAgICBjb25maWdfbm9kZSA9IE5vbmUKICAgIGtlcm5lbF9ub2RlID0gTm9uZQogICAgZm9yIG5vZGUgaW4gdHJlZS5ib2R5OgogICAgICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgYXN0LkFzc2lnbik6CiAgICAgICAgICAgIGlmIGNvbmZpZ19ub2RlIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiU3VibWlzc2lvbiBtdXN0IGRlZmluZSB7Q09ORklHU19OQU1FfSBleGFjdGx5IG9uY2UuIikKICAgICAgICAgICAgY29uZmlnX25vZGUgPSBub2RlCiAgICAgICAgZWxpZiBpc2luc3RhbmNlKG5vZGUsIGFzdC5GdW5jdGlvbkRlZik6CiAgICAgICAgICAgIGlmIGtlcm5lbF9ub2RlIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiU3VibWlzc2lvbiBtdXN0IGRlZmluZSB7S0VSTkVMX0ZVTkNUSU9OX05BTUV9IGV4YWN0bHkgb25jZS4iKQogICAgICAgICAgICBrZXJuZWxfbm9kZSA9IG5vZGUKICAgICAgICBlbHNlOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgICAgICBmIk9ubHkgYSB0b3AtbGV2ZWwge0NPTkZJR1NfTkFNRX0gYXNzaWdubWVudCBhbmQge0tFUk5FTF9GVU5DVElPTl9OQU1FfSBhcmUgYWxsb3dlZC4iCiAgICAgICAgICAgICkKCiAgICBpZiBjb25maWdfbm9kZSBpcyBOb25lOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlN1Ym1pc3Npb24gbXVzdCBkZWZpbmUgdG9wLWxldmVsIHtDT05GSUdTX05BTUV9LiIpCiAgICBpZiBrZXJuZWxfbm9kZSBpcyBOb25lOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlN1Ym1pc3Npb24gbXVzdCBkZWZpbmUgdG9wLWxldmVsIHtLRVJORUxfRlVOQ1RJT05fTkFNRX0uIikKCiAgICBpZiBsZW4oY29uZmlnX25vZGUudGFyZ2V0cykgIT0gMToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Q09ORklHU19OQU1FfSBtdXN0IGJlIGFzc2lnbmVkIGV4YWN0bHkgb25jZS4iKQogICAgaWYgbm90IGlzaW5zdGFuY2UoY29uZmlnX25vZGUudGFyZ2V0c1swXSwgYXN0Lk5hbWUpIG9yIGNvbmZpZ19ub2RlLnRhcmdldHNbMF0uaWQgIT0gQ09ORklHU19OQU1FOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlRvcC1sZXZlbCBhc3NpZ25tZW50IG11c3QgYmUgbmFtZWQge0NPTkZJR1NfTkFNRX0uIikKCiAgICB0cnk6CiAgICAgICAgY29uZmlnc192YWx1ZSA9IGFzdC5saXRlcmFsX2V2YWwoY29uZmlnX25vZGUudmFsdWUpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Q09ORklHU19OQU1FfSBtdXN0IGJlIGEgbGl0ZXJhbCBsaXN0IG9mIGRpY3Rpb25hcmllcy4iKSBmcm9tIGV4YwoKICAgIG9rLCBtZXNzYWdlLCBfID0gbm9ybWFsaXplX2NvbmZpZ3MoY29uZmlnc192YWx1ZSkKICAgIGlmIG5vdCBvazoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IobWVzc2FnZSkKCiAgICBpZiBrZXJuZWxfbm9kZS5uYW1lICE9IEtFUk5FTF9GVU5DVElPTl9OQU1FOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlRvcC1sZXZlbCBmdW5jdGlvbiBtdXN0IGJlIG5hbWVkIHtLRVJORUxfRlVOQ1RJT05fTkFNRX0uIikKICAgIGlmIGxlbihrZXJuZWxfbm9kZS5kZWNvcmF0b3JfbGlzdCkgIT0gMSBvciBub3QgX2lzX3RyaXRvbl9qaXRfZGVjb3JhdG9yKGtlcm5lbF9ub2RlLmRlY29yYXRvcl9saXN0WzBdKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7S0VSTkVMX0ZVTkNUSU9OX05BTUV9IG11c3QgYmUgZGVjb3JhdGVkIHdpdGggZXhhY3RseSBAdHJpdG9uLmppdC4iKQoKICAgIGFyZ19uYW1lcyA9IFthcmcuYXJnIGZvciBhcmcgaW4ga2VybmVsX25vZGUuYXJncy5hcmdzXQogICAgaWYgYXJnX25hbWVzICE9IEVYUEVDVEVEX0tFUk5FTF9BUkdTOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgZiJ7S0VSTkVMX0ZVTkNUSU9OX05BTUV9IG11c3QgaGF2ZSBleGFjdGx5IHRoaXMgc2lnbmF0dXJlOiAiCiAgICAgICAgICAgIGYieycsICcuam9pbihFWFBFQ1RFRF9LRVJORUxfQVJHUyl9IgogICAgICAgICkKCgpkZWYgbG9hZF9zdHVkZW50X3N1Ym1pc3Npb24ocGF0aDogUGF0aCk6CiAgICBzb3VyY2UgPSBwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgbmFtZXNwYWNlID0gewogICAgICAgICJfX2J1aWx0aW5zX18iOiBfX2J1aWx0aW5zX18sCiAgICAgICAgInRvcmNoIjogdG9yY2gsCiAgICAgICAgInRyaXRvbiI6IHRyaXRvbiwKICAgICAgICAidGwiOiB0bCwKICAgIH0KICAgIGV4ZWMoY29tcGlsZShzb3VyY2UsIHN0cihwYXRoKSwgImV4ZWMiKSwgbmFtZXNwYWNlLCBuYW1lc3BhY2UpCgogICAga2VybmVsID0gbmFtZXNwYWNlLmdldChLRVJORUxfRlVOQ1RJT05fTkFNRSkKICAgIGlmIGtlcm5lbCBpcyBOb25lIG9yIG5vdCBjYWxsYWJsZShrZXJuZWwpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIk1pc3NpbmcgY2FsbGFibGUge0tFUk5FTF9GVU5DVElPTl9OQU1FfSIpCgogICAgcmF3X2NvbmZpZ3MgPSBuYW1lc3BhY2UuZ2V0KENPTkZJR1NfTkFNRSkKICAgIGlmIHJhd19jb25maWdzIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTWlzc2luZyB7Q09ORklHU19OQU1FfSIpCgogICAgb2ssIG1lc3NhZ2UsIGNvbmZpZ3MgPSBub3JtYWxpemVfY29uZmlncyhyYXdfY29uZmlncykKICAgIGlmIG5vdCBvazoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IobWVzc2FnZSkKCiAgICByZXR1cm4ga2VybmVsLCBjb25maWdzCgoKZGVmIGJ1aWxkX3dyYXBwZXIoa2VybmVsLCBjb25maWc6IGRpY3Rbc3RyLCBpbnRdKToKICAgIGRlZiB3cmFwcGVkKGE6IHRvcmNoLlRlbnNvciwgYjogdG9yY2guVGVuc29yLCBjOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBtLCBrID0gYS5zaGFwZQogICAgICAgIGsyLCBuID0gYi5zaGFwZQogICAgICAgIGFzc2VydCBrID09IGsyCiAgICAgICAgYXNzZXJ0IGMuc2hhcGUgPT0gKG0sIG4pCiAgICAgICAgZCA9IHRvcmNoLmVtcHR5KChtLCBuKSwgZGV2aWNlPWEuZGV2aWNlLCBkdHlwZT10b3JjaC5mbG9hdDE2KQogICAgICAgIGdyaWQgPSAodHJpdG9uLmNkaXYobSwgY29uZmlnWyJCTE9DS19NIl0pLCB0cml0b24uY2RpdihuLCBjb25maWdbIkJMT0NLX04iXSkpCiAgICAgICAga2VybmVsW2dyaWRdKAogICAgICAgICAgICBhLAogICAgICAgICAgICBiLAogICAgICAgICAgICBjLAogICAgICAgICAgICBkLAogICAgICAgICAgICBtLAogICAgICAgICAgICBuLAogICAgICAgICAgICBrLAogICAgICAgICAgICBhLnN0cmlkZSgwKSwKICAgICAgICAgICAgYS5zdHJpZGUoMSksCiAgICAgICAgICAgIGIuc3RyaWRlKDApLAogICAgICAgICAgICBiLnN0cmlkZSgxKSwKICAgICAgICAgICAgYy5zdHJpZGUoMCksCiAgICAgICAgICAgIGMuc3RyaWRlKDEpLAogICAgICAgICAgICBkLnN0cmlkZSgwKSwKICAgICAgICAgICAgZC5zdHJpZGUoMSksCiAgICAgICAgICAgIEJMT0NLX009Y29uZmlnWyJCTE9DS19NIl0sCiAgICAgICAgICAgIEJMT0NLX049Y29uZmlnWyJCTE9DS19OIl0sCiAgICAgICAgICAgIEJMT0NLX0s9Y29uZmlnWyJCTE9DS19LIl0sCiAgICAgICAgICAgIG51bV93YXJwcz1jb25maWdbIm51bV93YXJwcyJdLAogICAgICAgICAgICBudW1fc3RhZ2VzPWNvbmZpZ1sibnVtX3N0YWdlcyJdLAogICAgICAgICkKICAgICAgICByZXR1cm4gZAoKICAgIHJldHVybiB3cmFwcGVkCgoKZGVmIHJlZmVyZW5jZV9tYXRtdWxfYWRkX3JlbHUoYTogdG9yY2guVGVuc29yLCBiOiB0b3JjaC5UZW5zb3IsIGM6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgcmV0dXJuIHRvcmNoLm1hdG11bChhLCBiKS5hZGQoYykucmVsdV8oKQoKCmRlZiB0aW1lX2tlcm5lbCgKICAgIGZuLAogICAgYTogdG9yY2guVGVuc29yLAogICAgYjogdG9yY2guVGVuc29yLAogICAgYzogdG9yY2guVGVuc29yLAogICAgd2FybXVwX2l0ZXJzOiBpbnQsCiAgICByZXBlYXRzOiBpbnQsCikgLT4gZmxvYXQ6CiAgICBfID0gZm4oYSwgYiwgYykKICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQoKICAgIGZvciBfIGluIHJhbmdlKHdhcm11cF9pdGVycyk6CiAgICAgICAgXyA9IGZuKGEsIGIsIGMpCiAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKCiAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGZvciBfIGluIHJhbmdlKHJlcGVhdHMpOgogICAgICAgIF8gPSBmbihhLCBiLCBjKQogICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpCiAgICByZXR1cm4gKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCkgLyByZXBlYXRzICogMTAwMC4wCgoKZGVmIHNlbGVjdF9iZXN0X2NvbmZpZyhrZXJuZWwsIGNvbmZpZ3M6IGxpc3RbZGljdFtzdHIsIGludF1dKSAtPiB0dXBsZVtkaWN0W3N0ciwgaW50XSwgZGljdF06CiAgICB0b3JjaC5tYW51YWxfc2VlZCgwKQogICAgYSA9IHRvcmNoLnJhbmRuKChNLCBLKSwgZGV2aWNlPSJjdWRhIiwgZHR5cGU9dG9yY2guZmxvYXQxNikKICAgIGIgPSB0b3JjaC5yYW5kbigoSywgTiksIGRldmljZT0iY3VkYSIsIGR0eXBlPXRvcmNoLmZsb2F0MTYpCiAgICBjID0gdG9yY2gucmFuZG4oKE0sIE4pLCBkZXZpY2U9ImN1ZGEiLCBkdHlwZT10b3JjaC5mbG9hdDE2KQoKICAgIHJlc3VsdHMgPSBbXQogICAgZmFpbHVyZXMgPSBbXQogICAgZm9yIGNvbmZpZyBpbiBjb25maWdzOgogICAgICAgIGZuID0gYnVpbGRfd3JhcHBlcihrZXJuZWwsIGNvbmZpZykKICAgICAgICB0cnk6CiAgICAgICAgICAgIHNlYXJjaF9tcyA9IHRpbWVfa2VybmVsKGZuLCBhLCBiLCBjLCBTRUFSQ0hfV0FSTVVQX0lURVJTLCBTRUFSQ0hfUkVQRUFUUykKICAgICAgICAgICAgcmVzdWx0cy5hcHBlbmQoeyJjb25maWciOiBjb25maWcsICJzZWFyY2hfbXMiOiBzZWFyY2hfbXN9KQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIGZhaWx1cmVzLmFwcGVuZCh7ImNvbmZpZyI6IGNvbmZpZywgInRyYWNlYmFjayI6IHRyYWNlYmFjay5mb3JtYXRfZXhjKCl9KQoKICAgIGlmIG5vdCByZXN1bHRzOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQWxsIHN1Ym1pdHRlZCBjb25maWdzIGZhaWxlZCBkdXJpbmcgY29uZmlnIHNlYXJjaC4iKQoKICAgIGJlc3QgPSBtaW4ocmVzdWx0cywga2V5PWxhbWJkYSBpdGVtOiBpdGVtWyJzZWFyY2hfbXMiXSkKICAgIHJldHVybiBiZXN0WyJjb25maWciXSwgewogICAgICAgICJ3YXJtdXBfaXRlcnMiOiBTRUFSQ0hfV0FSTVVQX0lURVJTLAogICAgICAgICJyZXBlYXRzIjogU0VBUkNIX1JFUEVBVFMsCiAgICAgICAgInJlc3VsdHMiOiByZXN1bHRzLAogICAgICAgICJmYWlsdXJlcyI6IGZhaWx1cmVzLAogICAgfQoKCmRlZiBjb3JyZWN0bmVzcyhmbikgLT4gZGljdDoKICAgIHRvcmNoLm1hbnVhbF9zZWVkKDApCiAgICBhID0gdG9yY2gucmFuZG4oKENPUlJFQ1RORVNTX1NIQVBFLCBDT1JSRUNUTkVTU19TSEFQRSksIGRldmljZT0iY3VkYSIsIGR0eXBlPXRvcmNoLmZsb2F0MTYpCiAgICBiID0gdG9yY2gucmFuZG4oKENPUlJFQ1RORVNTX1NIQVBFLCBDT1JSRUNUTkVTU19TSEFQRSksIGRldmljZT0iY3VkYSIsIGR0eXBlPXRvcmNoLmZsb2F0MTYpCiAgICBjID0gdG9yY2gucmFuZG4oKENPUlJFQ1RORVNTX1NIQVBFLCBDT1JSRUNUTkVTU19TSEFQRSksIGRldmljZT0iY3VkYSIsIGR0eXBlPXRvcmNoLmZsb2F0MTYpCgogICAgb3V0ID0gZm4oYSwgYiwgYykKICAgIHJlZiA9IHJlZmVyZW5jZV9tYXRtdWxfYWRkX3JlbHUoYSwgYiwgYykKICAgIG1heF9hYnNfZGlmZiA9IGZsb2F0KHRvcmNoLm1heCh0b3JjaC5hYnMob3V0IC0gcmVmKSkuaXRlbSgpKQogICAgb2sgPSB0b3JjaC5hbGxjbG9zZShvdXQsIHJlZiwgYXRvbD1BVE9MLCBydG9sPVJUT0wpCgogICAgcmV0dXJuIHsKICAgICAgICAib2siOiBib29sKG9rKSwKICAgICAgICAibWF4X2Fic19kaWZmIjogbWF4X2Fic19kaWZmLAogICAgICAgICJvdXRwdXRfZHR5cGUiOiBzdHIob3V0LmR0eXBlKSwKICAgIH0KCgpkZWYgYmVuY2htYXJrKGZuKSAtPiBkaWN0OgogICAgdG9yY2gubWFudWFsX3NlZWQoMCkKICAgIGEgPSB0b3JjaC5yYW5kbigoTSwgSyksIGRldmljZT0iY3VkYSIsIGR0eXBlPXRvcmNoLmZsb2F0MTYpCiAgICBiID0gdG9yY2gucmFuZG4oKEssIE4pLCBkZXZpY2U9ImN1ZGEiLCBkdHlwZT10b3JjaC5mbG9hdDE2KQogICAgYyA9IHRvcmNoLnJhbmRuKChNLCBOKSwgZGV2aWNlPSJjdWRhIiwgZHR5cGU9dG9yY2guZmxvYXQxNikKCiAgICBzdHVkZW50X21zID0gdGltZV9rZXJuZWwoZm4sIGEsIGIsIGMsIFdBUk1VUF9JVEVSUywgQkVOQ0hNQVJLX1JFUEVBVFMpCgogICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICBmb3IgXyBpbiByYW5nZShCRU5DSE1BUktfUkVQRUFUUyk6CiAgICAgICAgXyA9IHJlZmVyZW5jZV9tYXRtdWxfYWRkX3JlbHUoYSwgYiwgYykKICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQogICAgcmVmZXJlbmNlX21zID0gKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCkgLyBCRU5DSE1BUktfUkVQRUFUUyAqIDEwMDAuMAoKICAgIHJldHVybiB7CiAgICAgICAgInN0dWRlbnRfbXMiOiBzdHVkZW50X21zLAogICAgICAgICJyZWZlcmVuY2VfbXMiOiByZWZlcmVuY2VfbXMsCiAgICAgICAgInNwZWVkdXBfdnNfcHl0b3JjaCI6IHJlZmVyZW5jZV9tcyAvIHN0dWRlbnRfbXMsCiAgICAgICAgImJlbmNobWFya19zaGFwZSI6IFtNLCBOLCBLXSwKICAgICAgICAid2FybXVwX2l0ZXJzIjogV0FSTVVQX0lURVJTLAogICAgICAgICJyZXBlYXRzIjogQkVOQ0hNQVJLX1JFUEVBVFMsCiAgICB9CgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoCiAgICAgICAgZGVzY3JpcHRpb249IkxvY2FsIGNvcnJlY3RuZXNzIGFuZCBzcGVlZCB0ZXN0IGZvciBzdHVkZW50X2tlcm5lbC5weSIKICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoInN1Ym1pc3Npb24iLCBoZWxwPSJQYXRoIHRvIHN0dWRlbnRfa2VybmVsLnB5IikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgcGF0aCA9IFBhdGgoYXJncy5zdWJtaXNzaW9uKS5leHBhbmR1c2VyKCkucmVzb2x2ZSgpCiAgICBpZiBub3QgcGF0aC5leGlzdHMoKToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYiU3VibWlzc2lvbiBmaWxlIG5vdCBmb3VuZDoge3BhdGh9IikKICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkNVREEgaXMgcmVxdWlyZWQgZm9yIHRoaXMgdGVzdCBzY3JpcHQuIikKCiAgICB0cnk6CiAgICAgICAgdmFsaWRhdGVfc291cmNlKHBhdGgpCiAgICAgICAga2VybmVsLCBjb25maWdzID0gbG9hZF9zdHVkZW50X3N1Ym1pc3Npb24ocGF0aCkKICAgICAgICBzZWxlY3RlZF9jb25maWcsIGNvbmZpZ19zZWFyY2ggPSBzZWxlY3RfYmVzdF9jb25maWcoa2VybmVsLCBjb25maWdzKQogICAgICAgIGZuID0gYnVpbGRfd3JhcHBlcihrZXJuZWwsIHNlbGVjdGVkX2NvbmZpZykKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcHJpbnQodHJhY2ViYWNrLmZvcm1hdF9leGMoKSkKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCJTdWJtaXNzaW9uIHZhbGlkYXRpb24gb3IgY29uZmlnIHNlYXJjaCBmYWlsZWQuIikKCiAgICByZXN1bHQgPSB7CiAgICAgICAgInRvcmNoX3ZlcnNpb24iOiB0b3JjaC5fX3ZlcnNpb25fXywKICAgICAgICAiY3VkYV92ZXJzaW9uIjogdG9yY2gudmVyc2lvbi5jdWRhLAogICAgICAgICJkZXZpY2VfbmFtZSI6IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLAogICAgICAgICJzdWJtaXR0ZWRfY29uZmlncyI6IGNvbmZpZ3MsCiAgICAgICAgInNlbGVjdGVkX2NvbmZpZyI6IHNlbGVjdGVkX2NvbmZpZywKICAgICAgICAiY29uZmlnX3NlYXJjaCI6IGNvbmZpZ19zZWFyY2gsCiAgICB9CgogICAgdHJ5OgogICAgICAgIHJlc3VsdFsiY29ycmVjdG5lc3MiXSA9IGNvcnJlY3RuZXNzKGZuKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwcmludCh0cmFjZWJhY2suZm9ybWF0X2V4YygpKQogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkNvcnJlY3RuZXNzIGNoZWNrIGZhaWxlZCB3aXRoIGFuIGV4Y2VwdGlvbi4iKQoKICAgIGlmIG5vdCByZXN1bHRbImNvcnJlY3RuZXNzIl1bIm9rIl06CiAgICAgICAgcHJpbnQoanNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkpCiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiQ29ycmVjdG5lc3MgY2hlY2sgZmFpbGVkLiIpCgogICAgdHJ5OgogICAgICAgIHJlc3VsdFsiYmVuY2htYXJrIl0gPSBiZW5jaG1hcmsoZm4pCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHByaW50KHRyYWNlYmFjay5mb3JtYXRfZXhjKCkpCiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiQmVuY2htYXJrIGZhaWxlZCB3aXRoIGFuIGV4Y2VwdGlvbi4iKQoKICAgIHByaW50KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICB0cnk6CiAgICAgICAgbWFpbigpCiAgICBleGNlcHQgS2V5Ym9hcmRJbnRlcnJ1cHQ6CiAgICAgICAgc3lzLmV4aXQoMTMwKQo=',
    'student_submit.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHJlcXVlc3RzCgoKZGVmIF93cml0ZV9vdXRwdXQocGF0aF9zdHI6IHN0ciB8IE5vbmUsIHRleHQ6IHN0cikgLT4gTm9uZToKICAgIGlmIG5vdCBwYXRoX3N0cjoKICAgICAgICByZXR1cm4KICAgIFBhdGgocGF0aF9zdHIpLmV4cGFuZHVzZXIoKS53cml0ZV90ZXh0KHRleHQsIGVuY29kaW5nPSJ1dGYtOCIpCgoKZGVmIF9wcmludF9zdWJtaXRfc3VtbWFyeShwYXlsb2FkOiBkaWN0KSAtPiBOb25lOgogICAgcHJpbnQoIlN1Ym1pc3Npb24gYWNjZXB0ZWQuIikKICAgIHByaW50KGYiQ2FsbCBJRDoge3BheWxvYWRbJ2NhbGxfaWQnXX0iKQogICAgcHJpbnQoZiJGaWxlbmFtZToge3BheWxvYWRbJ2ZpbGVuYW1lJ119IikKICAgIHByaW50KGYiU2l6ZToge3BheWxvYWRbJ2ZpbGVfc2l6ZV9ieXRlcyddfSBieXRlcyIpCgoKZGVmIF9mb3JtYXRfY29uZmlnKGNvbmZpZzogZGljdCB8IE5vbmUpIC0+IHN0ciB8IE5vbmU6CiAgICBpZiBub3QgaXNpbnN0YW5jZShjb25maWcsIGRpY3QpOgogICAgICAgIHJldHVybiBOb25lCiAgICBvcmRlcmVkX2tleXMgPSBbIkJMT0NLX00iLCAiQkxPQ0tfTiIsICJCTE9DS19LIiwgIm51bV93YXJwcyIsICJudW1fc3RhZ2VzIl0KICAgIHBhcnRzID0gW10KICAgIGZvciBrZXkgaW4gb3JkZXJlZF9rZXlzOgogICAgICAgIGlmIGtleSBpbiBjb25maWc6CiAgICAgICAgICAgIHBhcnRzLmFwcGVuZChmIntrZXl9PXtjb25maWdba2V5XX0iKQogICAgaWYgbm90IHBhcnRzOgogICAgICAgIHJldHVybiBOb25lCiAgICByZXR1cm4gIiwgIi5qb2luKHBhcnRzKQoKCmRlZiBfcHJpbnRfd2FpdGluZ19zdGF0dXMoY2FsbF9pZDogc3RyLCBhdHRlbXB0OiBpbnQpIC0+IE5vbmU6CiAgICBkb3RzID0gIi4iICogKChhdHRlbXB0ICUgMykgKyAxKQogICAgbWVzc2FnZSA9IGYiXHJXYWl0aW5nIGZvciByZXN1bHR7ZG90czo8M30gY2FsbF9pZD17Y2FsbF9pZH0iCiAgICBwcmludChtZXNzYWdlLCBlbmQ9IiIsIGZsdXNoPVRydWUpCgoKZGVmIF9leHRyYWN0X3J1bm5lcl9wYXlsb2FkKHN0ZG91dDogc3RyKSAtPiBkaWN0IHwgTm9uZToKICAgIGZvciBsaW5lIGluIHJldmVyc2VkKHN0ZG91dC5zcGxpdGxpbmVzKCkpOgogICAgICAgIGNhbmRpZGF0ZSA9IGxpbmUuc3RyaXAoKQogICAgICAgIGlmIG5vdCBjYW5kaWRhdGU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdHJ5OgogICAgICAgICAgICBwYXJzZWQgPSBqc29uLmxvYWRzKGNhbmRpZGF0ZSkKICAgICAgICBleGNlcHQganNvbi5KU09ORGVjb2RlRXJyb3I6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgaXNpbnN0YW5jZShwYXJzZWQsIGRpY3QpOgogICAgICAgICAgICByZXR1cm4gcGFyc2VkCiAgICByZXR1cm4gTm9uZQoKCmRlZiBfbWVyZ2VkX2Vycm9yX3BheWxvYWQocGF5bG9hZDogZGljdCkgLT4gZGljdDoKICAgIHN0ZG91dCA9IHBheWxvYWQuZ2V0KCJzdGRvdXQiLCAiIikKICAgIG5lc3RlZCA9IF9leHRyYWN0X3J1bm5lcl9wYXlsb2FkKHN0ZG91dCkgaWYgc3Rkb3V0IGVsc2UgTm9uZQogICAgaWYgbm90IG5lc3RlZDoKICAgICAgICByZXR1cm4gcGF5bG9hZAoKICAgIG1lcmdlZCA9IGRpY3QocGF5bG9hZCkKICAgIG1lcmdlZC51cGRhdGUobmVzdGVkKQogICAgbWVyZ2VkWyJzdGRvdXQiXSA9IHBheWxvYWQuZ2V0KCJzdGRvdXQiLCAiIikKICAgIG1lcmdlZFsic3RkZXJyIl0gPSBwYXlsb2FkLmdldCgic3RkZXJyIiwgIiIpCiAgICByZXR1cm4gbWVyZ2VkCgoKZGVmIF9sYXN0X2Vycm9yX2xpbmUodHJhY2ViYWNrX3RleHQ6IHN0cikgLT4gc3RyIHwgTm9uZToKICAgIGZvciBsaW5lIGluIHJldmVyc2VkKHRyYWNlYmFja190ZXh0LnNwbGl0bGluZXMoKSk6CiAgICAgICAgc3RyaXBwZWQgPSBsaW5lLnN0cmlwKCkKICAgICAgICBpZiBub3Qgc3RyaXBwZWQgb3Igc3RyaXBwZWQgPT0gIl4iOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIHN0cmlwcGVkLnN0YXJ0c3dpdGgoIlRyYWNlYmFjayAiKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICByZXR1cm4gc3RyaXBwZWQKICAgIHJldHVybiBOb25lCgoKZGVmIF9leHRyYWN0X2NvbXBpbGF0aW9uX2Jsb2NrKHRyYWNlYmFja190ZXh0OiBzdHIpIC0+IHN0ciB8IE5vbmU6CiAgICBsaW5lcyA9IHRyYWNlYmFja190ZXh0LnNwbGl0bGluZXMoKQogICAgc3RhcnQgPSBOb25lCiAgICBmb3IgaSwgbGluZSBpbiBlbnVtZXJhdGUobGluZXMpOgogICAgICAgIGlmICJDb21waWxhdGlvbkVycm9yOiIgaW4gbGluZToKICAgICAgICAgICAgc3RhcnQgPSBpCiAgICAgICAgICAgIGJyZWFrCgogICAgaWYgc3RhcnQgaXMgTm9uZToKICAgICAgICByZXR1cm4gTm9uZQoKICAgIGJsb2NrID0gIlxuIi5qb2luKGxpbmVzW3N0YXJ0Ol0pLnN0cmlwKCkKICAgIHJldHVybiBibG9jayBvciBOb25lCgoKZGVmIF9mcmllbmRseV9oaW50KGVycm9yX3BheWxvYWQ6IGRpY3QpIC0+IHN0ciB8IE5vbmU6CiAgICBjb21iaW5lZCA9ICJcbiIuam9pbigKICAgICAgICBzdHIocGFydCkKICAgICAgICBmb3IgcGFydCBpbiAoCiAgICAgICAgICAgIGVycm9yX3BheWxvYWQuZ2V0KCJtZXNzYWdlIiwgIiIpLAogICAgICAgICAgICBlcnJvcl9wYXlsb2FkLmdldCgidHJhY2ViYWNrIiwgIiIpLAogICAgICAgICAgICBlcnJvcl9wYXlsb2FkLmdldCgic3RkZXJyIiwgIiIpLAogICAgICAgICAgICBlcnJvcl9wYXlsb2FkLmdldCgic3Rkb3V0IiwgIiIpLAogICAgICAgICkKICAgICAgICBpZiBwYXJ0CiAgICApCgogICAgaWYgInplcm9zKCkgbWlzc2luZyAxIHJlcXVpcmVkIHBvc2l0aW9uYWwgYXJndW1lbnQ6ICdkdHlwZSciIGluIGNvbWJpbmVkOgogICAgICAgIHJldHVybiAoCiAgICAgICAgICAgICJgdGwuemVyb3NgIG5lZWRzIGFuIGV4cGxpY2l0IGR0eXBlLiBGb3IgdGhlIGFjY3VtdWxhdG9yLCB3cml0ZSAiCiAgICAgICAgICAgICJgdGwuemVyb3MoKEJMT0NLX00sIEJMT0NLX04pLCBkdHlwZT10bC5mbG9hdDMyKWAuIgogICAgICAgICkKCiAgICBpZiAiVG9wLWxldmVsIGZ1bmN0aW9uIG11c3QgYmUgbmFtZWQiIGluIGNvbWJpbmVkOgogICAgICAgIHJldHVybiAiQ2hlY2sgdGhhdCB5b3VyIGZpbGUgb25seSBkZWZpbmVzIGBtYXRtdWxfYWRkX3JlbHVfa2VybmVsX2ZwMTZgLiIKCiAgICBpZiAibXVzdCBiZSBkZWNvcmF0ZWQgd2l0aCBleGFjdGx5IEB0cml0b24uaml0IiBpbiBjb21iaW5lZDoKICAgICAgICByZXR1cm4gIkFkZCBleGFjdGx5IG9uZSBkZWNvcmF0b3I6IGBAdHJpdG9uLmppdGAuIgoKICAgIGlmICJtdXN0IGhhdmUgZXhhY3RseSB0aGlzIHNpZ25hdHVyZSIgaW4gY29tYmluZWQ6CiAgICAgICAgcmV0dXJuICJNYXRjaCB0aGUgcmVxdWlyZWQga2VybmVsIGFyZ3VtZW50IGxpc3QgZXhhY3RseSwgaW5jbHVkaW5nIGFyZ3VtZW50IG9yZGVyLiIKCiAgICBpZiAiU3VibWlzc2lvbiBtdXN0IGNvbnRhaW4gZXhhY3RseSB0d28gdG9wLWxldmVsIGRlZmluaXRpb25zIiBpbiBjb21iaW5lZDoKICAgICAgICByZXR1cm4gIllvdXIgZmlsZSBzaG91bGQgY29udGFpbiBvbmx5IGBLRVJORUxfQ09ORklHUyA9IFsuLi5dYCBhbmQgYG1hdG11bF9hZGRfcmVsdV9rZXJuZWxfZnAxNmAuIgoKICAgIGlmICJtdXN0IGRlZmluZSB0b3AtbGV2ZWwgS0VSTkVMX0NPTkZJR1MiIGluIGNvbWJpbmVkIG9yICJtdXN0IGRlZmluZSBLRVJORUxfQ09ORklHUyIgaW4gY29tYmluZWQ6CiAgICAgICAgcmV0dXJuICJBZGQgYSB0b3AtbGV2ZWwgYEtFUk5FTF9DT05GSUdTID0gWy4uLl1gIGxpc3QgYmVmb3JlIHRoZSBrZXJuZWwuIgoKICAgIGlmICJLRVJORUxfQ09ORklHUyBtdXN0IGNvbnRhaW4gYmV0d2VlbiIgaW4gY29tYmluZWQ6CiAgICAgICAgcmV0dXJuICJTdWJtaXQgYSBzaG9ydCBjYW5kaWRhdGUgbGlzdCwgZS5nLiAxIHRvIDUgY29uZmlncy4iCgogICAgaWYgIm11c3QgY29udGFpbiBleGFjdGx5IHRoZXNlIGtleXMiIGluIGNvbWJpbmVkOgogICAgICAgIHJldHVybiAiRWFjaCBjb25maWcgbXVzdCBpbmNsdWRlIEJMT0NLX00sIEJMT0NLX04sIEJMT0NLX0ssIG51bV93YXJwcywgYW5kIG51bV9zdGFnZXMuIgoKICAgIHJldHVybiBOb25lCgoKZGVmIF9wcmludF9ibG9jayh0aXRsZTogc3RyLCBib2R5OiBzdHIpIC0+IE5vbmU6CiAgICBwcmludChmIlxue3RpdGxlfSIpCiAgICBwcmludChib2R5LnJzdHJpcCgpKQoKCmRlZiBfcHJpbnRfaHR0cF9lcnJvcihyZXNwb25zZTogcmVxdWVzdHMuUmVzcG9uc2UpIC0+IE5vbmU6CiAgICB0cnk6CiAgICAgICAgcGF5bG9hZCA9IHJlc3BvbnNlLmpzb24oKQogICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgcHJpbnQocmVzcG9uc2UudGV4dCkKICAgICAgICByZXR1cm4KCiAgICBpZiBpc2luc3RhbmNlKHBheWxvYWQsIGRpY3QpOgogICAgICAgIGRldGFpbCA9IHBheWxvYWQuZ2V0KCJkZXRhaWwiKSBvciBwYXlsb2FkLmdldCgibWVzc2FnZSIpCiAgICAgICAgaWYgZGV0YWlsOgogICAgICAgICAgICBwcmludChmIkVycm9yOiB7ZGV0YWlsfSIpCiAgICAgICAgICAgIHJldHVybgoKICAgIHByaW50KGpzb24uZHVtcHMocGF5bG9hZCwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSkKCgpkZWYgX2h0dHBfZXJyb3JfcGF5bG9hZChyZXNwb25zZTogcmVxdWVzdHMuUmVzcG9uc2UpIC0+IGRpY3Q6CiAgICBwYXlsb2FkOiBkaWN0W3N0ciwgb2JqZWN0XSA9IHsKICAgICAgICAiaHR0cF9zdGF0dXMiOiByZXNwb25zZS5zdGF0dXNfY29kZSwKICAgIH0KICAgIHRyeToKICAgICAgICBib2R5ID0gcmVzcG9uc2UuanNvbigpCiAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICBwYXlsb2FkWyJib2R5Il0gPSByZXNwb25zZS50ZXh0CiAgICAgICAgcmV0dXJuIHBheWxvYWQKCiAgICBpZiBpc2luc3RhbmNlKGJvZHksIGRpY3QpOgogICAgICAgIHBheWxvYWQudXBkYXRlKGJvZHkpCiAgICBlbHNlOgogICAgICAgIHBheWxvYWRbImJvZHkiXSA9IGJvZHkKICAgIHJldHVybiBwYXlsb2FkCgoKZGVmIF9wcmludF9yZXN1bHRfc3VtbWFyeShwYXlsb2FkOiBkaWN0KSAtPiBOb25lOgogICAgc3RhdHVzID0gcGF5bG9hZC5nZXQoInN0YXR1cyIsICJ1bmtub3duIikKICAgIHByaW50KGYiU3RhdHVzOiB7c3RhdHVzfSIpCgogICAgaWYgc3RhdHVzICE9ICJvayI6CiAgICAgICAgZXJyb3JfcGF5bG9hZCA9IF9tZXJnZWRfZXJyb3JfcGF5bG9hZChwYXlsb2FkKQoKICAgICAgICBtZXNzYWdlID0gZXJyb3JfcGF5bG9hZC5nZXQoIm1lc3NhZ2UiKQogICAgICAgIGlmIG1lc3NhZ2U6CiAgICAgICAgICAgIHByaW50KGYiTWVzc2FnZToge21lc3NhZ2V9IikKCiAgICAgICAgY29ycmVjdG5lc3MgPSBlcnJvcl9wYXlsb2FkLmdldCgiY29ycmVjdG5lc3MiKQogICAgICAgIGlmIGlzaW5zdGFuY2UoY29ycmVjdG5lc3MsIGRpY3QpOgogICAgICAgICAgICBwcmludChmIlBhc3NlZCBjb3JyZWN0bmVzczogeyd5ZXMnIGlmIGNvcnJlY3RuZXNzLmdldCgnb2snKSBlbHNlICdubyd9IikKICAgICAgICAgICAgaWYgY29ycmVjdG5lc3MuZ2V0KCJtYXhfYWJzX2RpZmYiKSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHByaW50KGYiTWF4IGFicyBkaWZmOiB7Y29ycmVjdG5lc3NbJ21heF9hYnNfZGlmZiddfSIpCgogICAgICAgIHRyYWNlYmFja190ZXh0ID0gZXJyb3JfcGF5bG9hZC5nZXQoInRyYWNlYmFjayIsICIiKQogICAgICAgIGlmIHRyYWNlYmFja190ZXh0OgogICAgICAgICAgICBtYWluX2Vycm9yID0gX2xhc3RfZXJyb3JfbGluZSh0cmFjZWJhY2tfdGV4dCkKICAgICAgICAgICAgaWYgbWFpbl9lcnJvcjoKICAgICAgICAgICAgICAgIHByaW50KGYiTWFpbiBlcnJvcjoge21haW5fZXJyb3J9IikKCiAgICAgICAgICAgIGNvbXBpbGF0aW9uX2Jsb2NrID0gX2V4dHJhY3RfY29tcGlsYXRpb25fYmxvY2sodHJhY2ViYWNrX3RleHQpCiAgICAgICAgICAgIGlmIGNvbXBpbGF0aW9uX2Jsb2NrOgogICAgICAgICAgICAgICAgX3ByaW50X2Jsb2NrKCJUcml0b24gY29tcGlsZXIgb3V0cHV0OiIsIGNvbXBpbGF0aW9uX2Jsb2NrKQoKICAgICAgICBoaW50ID0gX2ZyaWVuZGx5X2hpbnQoZXJyb3JfcGF5bG9hZCkKICAgICAgICBpZiBoaW50OgogICAgICAgICAgICBwcmludChmIkxpa2VseSBmaXg6IHtoaW50fSIpCgogICAgICAgIHN0ZGVyciA9IGVycm9yX3BheWxvYWQuZ2V0KCJzdGRlcnIiLCAiIikKICAgICAgICBpZiBzdGRlcnI6CiAgICAgICAgICAgIF9wcmludF9ibG9jaygic3RkZXJyOiIsIHN0ZGVycikKCiAgICAgICAgaWYgbm90IHRyYWNlYmFja190ZXh0OgogICAgICAgICAgICBzdGRvdXQgPSBlcnJvcl9wYXlsb2FkLmdldCgic3Rkb3V0IiwgIiIpCiAgICAgICAgICAgIGlmIHN0ZG91dDoKICAgICAgICAgICAgICAgIG5lc3RlZCA9IF9leHRyYWN0X3J1bm5lcl9wYXlsb2FkKHN0ZG91dCkKICAgICAgICAgICAgICAgIGlmIG5vdCBuZXN0ZWQ6CiAgICAgICAgICAgICAgICAgICAgX3ByaW50X2Jsb2NrKCJzdGRvdXQ6Iiwgc3Rkb3V0KQoKICAgICAgICBwcmludCgiVXNlIGAtLWpzb25gIGlmIHlvdSB3YW50IHRoZSBmdWxsIHJhdyBwYXlsb2FkLiIpCiAgICAgICAgcmV0dXJuCgogICAgY29ycmVjdG5lc3MgPSBwYXlsb2FkLmdldCgiY29ycmVjdG5lc3MiLCB7fSkKICAgIHBhc3NlZCA9IGJvb2woY29ycmVjdG5lc3MuZ2V0KCJvayIpKQogICAgcHJpbnQoZiJQYXNzZWQgY29ycmVjdG5lc3M6IHsneWVzJyBpZiBwYXNzZWQgZWxzZSAnbm8nfSIpCgogICAgaWYgY29ycmVjdG5lc3MuZ2V0KCJtYXhfYWJzX2RpZmYiKSBpcyBub3QgTm9uZToKICAgICAgICBwcmludChmIk1heCBhYnMgZGlmZjoge2NvcnJlY3RuZXNzWydtYXhfYWJzX2RpZmYnXTouNmZ9IikKCiAgICBzZWxlY3RlZF9jb25maWcgPSBwYXlsb2FkLmdldCgic2VsZWN0ZWRfY29uZmlnIikKICAgIHNlbGVjdGVkX2NvbmZpZ190ZXh0ID0gX2Zvcm1hdF9jb25maWcoc2VsZWN0ZWRfY29uZmlnKQogICAgaWYgc2VsZWN0ZWRfY29uZmlnX3RleHQ6CiAgICAgICAgcHJpbnQoZiJTZWxlY3RlZCBjb25maWc6IHtzZWxlY3RlZF9jb25maWdfdGV4dH0iKQoKICAgIHN1Ym1pdHRlZF9jb25maWdzID0gcGF5bG9hZC5nZXQoInN1Ym1pdHRlZF9jb25maWdzIikKICAgIGlmIGlzaW5zdGFuY2Uoc3VibWl0dGVkX2NvbmZpZ3MsIGxpc3QpOgogICAgICAgIHByaW50KGYiU3VibWl0dGVkIGNvbmZpZ3M6IHtsZW4oc3VibWl0dGVkX2NvbmZpZ3MpfSIpCgogICAgc3R1ZGVudF9tcyA9IHBheWxvYWQuZ2V0KCJzdHVkZW50X21zIikKICAgIHJlZmVyZW5jZV9tcyA9IHBheWxvYWQuZ2V0KCJyZWZlcmVuY2VfbXMiKQogICAgc3BlZWR1cCA9IHBheWxvYWQuZ2V0KCJzcGVlZHVwX3ZzX3B5dG9yY2giKQoKICAgIGlmIHN0dWRlbnRfbXMgaXMgbm90IE5vbmU6CiAgICAgICAgcHJpbnQoZiJZb3VyIGtlcm5lbDoge3N0dWRlbnRfbXM6LjRmfSBtcyIpCiAgICBpZiByZWZlcmVuY2VfbXMgaXMgbm90IE5vbmU6CiAgICAgICAgcHJpbnQoZiJQeVRvcmNoIHJlZjoge3JlZmVyZW5jZV9tczouNGZ9IG1zIikKICAgIGlmIHNwZWVkdXAgaXMgbm90IE5vbmU6CiAgICAgICAgcHJpbnQoZiJTcGVlZHVwIHZzIHJlZjoge3NwZWVkdXA6LjRmfXgiKQoKICAgIGRldmljZV9uYW1lID0gcGF5bG9hZC5nZXQoImRldmljZV9uYW1lIikKICAgIGlmIGRldmljZV9uYW1lOgogICAgICAgIHByaW50KGYiRGV2aWNlOiB7ZGV2aWNlX25hbWV9IikKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigKICAgICAgICBkZXNjcmlwdGlvbj0iVXBsb2FkIGEgVHJpdG9uIHN1Ym1pc3Npb24gdG8gdGhlIGNvdXJzZSBNb2RhbCBncmFkZXIgYW5kIHBvbGwgZm9yIHRoZSByZXN1bHQuIgogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgic3VibWlzc2lvbiIsIGhlbHA9IlBhdGggdG8gc3R1ZGVudF9rZXJuZWwucHkiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1iYXNlLXVybCIsCiAgICAgICAgZGVmYXVsdD1vcy5lbnZpcm9uLmdldCgiR1JBREVSX0JBU0VfVVJMIiwgIiIpLnJzdHJpcCgiLyIpLAogICAgICAgIGhlbHA9IkJhc2UgVVJMIGZvciB0aGUgZGVwbG95ZWQgZ3JhZGVyLCBlLmcuIGh0dHBzOi8vPHdvcmtzcGFjZT4tLXdlYi1hcHAubW9kYWwucnVuIiwKICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoCiAgICAgICAgIi0tdG9rZW4iLAogICAgICAgIGRlZmF1bHQ9b3MuZW52aXJvbi5nZXQoIkdSQURFUl9UT0tFTiIsICIiKSwKICAgICAgICBoZWxwPSJDb3Vyc2UgZ3JhZGVyIHRva2VuLiBEZWZhdWx0cyB0byBHUkFERVJfVE9LRU4uIiwKICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoCiAgICAgICAgIi0tcG9sbC1pbnRlcnZhbCIsCiAgICAgICAgdHlwZT1mbG9hdCwKICAgICAgICBkZWZhdWx0PTIuMCwKICAgICAgICBoZWxwPSJTZWNvbmRzIGJldHdlZW4gcG9sbGluZyBhdHRlbXB0cy4iLAogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1zdWJtaXQtdGltZW91dCIsCiAgICAgICAgdHlwZT1mbG9hdCwKICAgICAgICBkZWZhdWx0PTE4MC4wLAogICAgICAgIGhlbHA9IlJlYWQgdGltZW91dCBpbiBzZWNvbmRzIGZvciB0aGUgaW5pdGlhbCAvc3VibWl0IHJlcXVlc3QuIiwKICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoCiAgICAgICAgIi0tcG9sbC10aW1lb3V0IiwKICAgICAgICB0eXBlPWZsb2F0LAogICAgICAgIGRlZmF1bHQ9NjAuMCwKICAgICAgICBoZWxwPSJSZWFkIHRpbWVvdXQgaW4gc2Vjb25kcyBmb3IgZWFjaCAvcmVzdWx0IHBvbGwgcmVxdWVzdC4iLAogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1vdXRwdXQiLAogICAgICAgIGRlZmF1bHQ9IiIsCiAgICAgICAgaGVscD0iT3B0aW9uYWwgZmlsZSB0byB3cml0ZSB0aGUgZmluYWwgb3V0cHV0IHBheWxvYWQgdG8uIiwKICAgICkKICAgIG91dHB1dF9ncm91cCA9IHBhcnNlci5hZGRfbXV0dWFsbHlfZXhjbHVzaXZlX2dyb3VwKCkKICAgIG91dHB1dF9ncm91cC5hZGRfYXJndW1lbnQoCiAgICAgICAgIi0tanNvbiIsCiAgICAgICAgZGVzdD0ianNvbl9vdXRwdXQiLAogICAgICAgIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCiAgICAgICAgaGVscD0iUHJpbnQgbWFjaGluZS1yZWFkYWJsZSBKU09OIG91dHB1dC4iLAogICAgKQogICAgb3V0cHV0X2dyb3VwLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1odW1hbiIsCiAgICAgICAgZGVzdD0ianNvbl9vdXRwdXQiLAogICAgICAgIGFjdGlvbj0ic3RvcmVfZmFsc2UiLAogICAgICAgIGhlbHA9IlByaW50IGEgaHVtYW4tcmVhZGFibGUgc3VtbWFyeS4gVGhpcyBpcyB0aGUgZGVmYXVsdC4iLAogICAgKQogICAgcGFyc2VyLnNldF9kZWZhdWx0cyhqc29uX291dHB1dD1GYWxzZSkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgc3VibWlzc2lvbl9wYXRoID0gUGF0aChhcmdzLnN1Ym1pc3Npb24pLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgIGlmIG5vdCBzdWJtaXNzaW9uX3BhdGguZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChmIlN1Ym1pc3Npb24gZmlsZSBub3QgZm91bmQ6IHtzdWJtaXNzaW9uX3BhdGh9IikKICAgIGlmIHN1Ym1pc3Npb25fcGF0aC5zdWZmaXggIT0gIi5weSI6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiU3VibWlzc2lvbiBtdXN0IGJlIGEgLnB5IGZpbGUiKQogICAgaWYgbm90IGFyZ3MuYmFzZV91cmw6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiUHJvdmlkZSAtLWJhc2UtdXJsIG9yIHNldCBHUkFERVJfQkFTRV9VUkwiKQogICAgaWYgbm90IGFyZ3MudG9rZW46CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiUHJvdmlkZSAtLXRva2VuIG9yIHNldCBHUkFERVJfVE9LRU4iKQoKICAgIGhlYWRlcnMgPSB7CiAgICAgICAgIlgtQ1NFMjkxLURTQzI5MS1Ub2tlbiI6IGFyZ3MudG9rZW4sCiAgICB9CgogICAgd2FybXVwX3BheWxvYWQ6IGRpY3Rbc3RyLCBvYmplY3RdCgogICAgIyBCZXN0LWVmZm9ydCB3YXJtdXAgdG8gcmVkdWNlIGZpcnN0LXJlcXVlc3QgY29sZCBzdGFydCBsYXRlbmN5LgogICAgdHJ5OgogICAgICAgIHdhcm11cCA9IHJlcXVlc3RzLmdldCgKICAgICAgICAgICAgZiJ7YXJncy5iYXNlX3VybH0vaGVhbHRoeiIsCiAgICAgICAgICAgIGhlYWRlcnM9aGVhZGVycywKICAgICAgICAgICAgdGltZW91dD0oMTAsIDYwKSwKICAgICAgICApCiAgICAgICAgd2FybXVwX3BheWxvYWQgPSB7Im9rIjogVHJ1ZSwgImh0dHBfc3RhdHVzIjogd2FybXVwLnN0YXR1c19jb2RlfQogICAgICAgIGlmIG5vdCBhcmdzLmpzb25fb3V0cHV0OgogICAgICAgICAgICBwcmludChmIlt3YXJtdXBdIC9oZWFsdGh6IC0+IEhUVFAge3dhcm11cC5zdGF0dXNfY29kZX0iKQogICAgZXhjZXB0IHJlcXVlc3RzLlJlcXVlc3RFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIHdhcm11cF9wYXlsb2FkID0gewogICAgICAgICAgICAib2siOiBGYWxzZSwKICAgICAgICAgICAgImVycm9yX3R5cGUiOiB0eXBlKGV4YykuX19uYW1lX18sCiAgICAgICAgICAgICJtZXNzYWdlIjogc3RyKGV4YyksCiAgICAgICAgfQogICAgICAgIGlmIG5vdCBhcmdzLmpzb25fb3V0cHV0OgogICAgICAgICAgICBwcmludChmIlt3YXJtdXBdIHNraXBwZWQ6IHt0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfSIpCgogICAgd2l0aCBzdWJtaXNzaW9uX3BhdGgub3BlbigicmIiKSBhcyBmOgogICAgICAgIHJlc3BvbnNlID0gcmVxdWVzdHMucG9zdCgKICAgICAgICAgICAgZiJ7YXJncy5iYXNlX3VybH0vc3VibWl0IiwKICAgICAgICAgICAgaGVhZGVycz1oZWFkZXJzLAogICAgICAgICAgICBmaWxlcz17ImZpbGUiOiAoc3VibWlzc2lvbl9wYXRoLm5hbWUsIGYsICJ0ZXh0L3gtcHl0aG9uIil9LAogICAgICAgICAgICB0aW1lb3V0PSgxMCwgYXJncy5zdWJtaXRfdGltZW91dCksCiAgICAgICAgKQoKICAgIGlmIHJlc3BvbnNlLnN0YXR1c19jb2RlID49IDQwMDoKICAgICAgICBpZiBhcmdzLmpzb25fb3V0cHV0OgogICAgICAgICAgICBvdXRwdXRfdGV4dCA9IGpzb24uZHVtcHMoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgInN0YXR1cyI6ICJzdWJtaXRfaHR0cF9lcnJvciIsCiAgICAgICAgICAgICAgICAgICAgIndhcm11cCI6IHdhcm11cF9wYXlsb2FkLAogICAgICAgICAgICAgICAgICAgICJzdWJtaXRfZXJyb3IiOiBfaHR0cF9lcnJvcl9wYXlsb2FkKHJlc3BvbnNlKSwKICAgICAgICAgICAgICAgIH0sCiAgICAgICAgICAgICAgICBpbmRlbnQ9MiwKICAgICAgICAgICAgICAgIHNvcnRfa2V5cz1UcnVlLAogICAgICAgICAgICApCiAgICAgICAgICAgIF93cml0ZV9vdXRwdXQoYXJncy5vdXRwdXQsIG91dHB1dF90ZXh0ICsgIlxuIikKICAgICAgICAgICAgcHJpbnQob3V0cHV0X3RleHQpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgX3ByaW50X2h0dHBfZXJyb3IocmVzcG9uc2UpCiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChmIlN1Ym1pdCBmYWlsZWQgd2l0aCBIVFRQIHtyZXNwb25zZS5zdGF0dXNfY29kZX0iKQoKICAgIHN1Ym1pdF9wYXlsb2FkID0gcmVzcG9uc2UuanNvbigpCiAgICBjYWxsX2lkID0gc3VibWl0X3BheWxvYWRbImNhbGxfaWQiXQogICAgaWYgbm90IGFyZ3MuanNvbl9vdXRwdXQ6CiAgICAgICAgX3ByaW50X3N1Ym1pdF9zdW1tYXJ5KHN1Ym1pdF9wYXlsb2FkKQoKICAgIHBvbGxfYXR0ZW1wdCA9IDAKICAgIHdoaWxlIFRydWU6CiAgICAgICAgcG9sbF9yZXNwb25zZSA9IHJlcXVlc3RzLmdldCgKICAgICAgICAgICAgZiJ7YXJncy5iYXNlX3VybH0vcmVzdWx0L3tjYWxsX2lkfSIsCiAgICAgICAgICAgIGhlYWRlcnM9aGVhZGVycywKICAgICAgICAgICAgdGltZW91dD0oMTAsIGFyZ3MucG9sbF90aW1lb3V0KSwKICAgICAgICApCiAgICAgICAgaWYgcG9sbF9yZXNwb25zZS5zdGF0dXNfY29kZSA9PSAyMDI6CiAgICAgICAgICAgIGlmIG5vdCBhcmdzLmpzb25fb3V0cHV0OgogICAgICAgICAgICAgICAgX3ByaW50X3dhaXRpbmdfc3RhdHVzKGNhbGxfaWQsIHBvbGxfYXR0ZW1wdCkKICAgICAgICAgICAgcG9sbF9hdHRlbXB0ICs9IDEKICAgICAgICAgICAgdGltZS5zbGVlcChhcmdzLnBvbGxfaW50ZXJ2YWwpCiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIGlmIHBvbGxfcmVzcG9uc2Uuc3RhdHVzX2NvZGUgPj0gNDAwOgogICAgICAgICAgICBpZiBub3QgYXJncy5qc29uX291dHB1dCBhbmQgcG9sbF9hdHRlbXB0ID4gMDoKICAgICAgICAgICAgICAgIHByaW50KCkKICAgICAgICAgICAgaWYgYXJncy5qc29uX291dHB1dDoKICAgICAgICAgICAgICAgIG91dHB1dF90ZXh0ID0ganNvbi5kdW1wcygKICAgICAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgICAgICJzdGF0dXMiOiAicmVzdWx0X2h0dHBfZXJyb3IiLAogICAgICAgICAgICAgICAgICAgICAgICAid2FybXVwIjogd2FybXVwX3BheWxvYWQsCiAgICAgICAgICAgICAgICAgICAgICAgICJzdWJtaXQiOiBzdWJtaXRfcGF5bG9hZCwKICAgICAgICAgICAgICAgICAgICAgICAgInJlc3VsdF9lcnJvciI6IF9odHRwX2Vycm9yX3BheWxvYWQocG9sbF9yZXNwb25zZSksCiAgICAgICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgICAgICBpbmRlbnQ9MiwKICAgICAgICAgICAgICAgICAgICBzb3J0X2tleXM9VHJ1ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIF93cml0ZV9vdXRwdXQoYXJncy5vdXRwdXQsIG91dHB1dF90ZXh0ICsgIlxuIikKICAgICAgICAgICAgICAgIHByaW50KG91dHB1dF90ZXh0KQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgX3ByaW50X2h0dHBfZXJyb3IocG9sbF9yZXNwb25zZSkKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChmIlBvbGxpbmcgZmFpbGVkIHdpdGggSFRUUCB7cG9sbF9yZXNwb25zZS5zdGF0dXNfY29kZX0iKQoKICAgICAgICByZXN1bHRfcGF5bG9hZCA9IHBvbGxfcmVzcG9uc2UuanNvbigpCiAgICAgICAgaWYgYXJncy5qc29uX291dHB1dDoKICAgICAgICAgICAgb3V0cHV0X3RleHQgPSBqc29uLmR1bXBzKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJzdGF0dXMiOiAib2siLAogICAgICAgICAgICAgICAgICAgICJ3YXJtdXAiOiB3YXJtdXBfcGF5bG9hZCwKICAgICAgICAgICAgICAgICAgICAic3VibWl0Ijogc3VibWl0X3BheWxvYWQsCiAgICAgICAgICAgICAgICAgICAgInJlc3VsdCI6IHJlc3VsdF9wYXlsb2FkLAogICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgIGluZGVudD0yLAogICAgICAgICAgICAgICAgc29ydF9rZXlzPVRydWUsCiAgICAgICAgICAgICkKICAgICAgICAgICAgX3dyaXRlX291dHB1dChhcmdzLm91dHB1dCwgb3V0cHV0X3RleHQgKyAiXG4iKQogICAgICAgICAgICBwcmludChvdXRwdXRfdGV4dCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBpZiBwb2xsX2F0dGVtcHQgPiAwOgogICAgICAgICAgICAgICAgcHJpbnQoKQogICAgICAgICAgICBwcmludCgpCiAgICAgICAgICAgIGZyb20gaW8gaW1wb3J0IFN0cmluZ0lPCiAgICAgICAgICAgIGZyb20gY29udGV4dGxpYiBpbXBvcnQgcmVkaXJlY3Rfc3Rkb3V0CgogICAgICAgICAgICBidWYgPSBTdHJpbmdJTygpCiAgICAgICAgICAgIHdpdGggcmVkaXJlY3Rfc3Rkb3V0KGJ1Zik6CiAgICAgICAgICAgICAgICBfcHJpbnRfcmVzdWx0X3N1bW1hcnkocmVzdWx0X3BheWxvYWQpCiAgICAgICAgICAgIG91dHB1dF90ZXh0ID0gYnVmLmdldHZhbHVlKCkKICAgICAgICAgICAgX3dyaXRlX291dHB1dChhcmdzLm91dHB1dCwgb3V0cHV0X3RleHQpCiAgICAgICAgICAgIHByaW50KG91dHB1dF90ZXh0LCBlbmQ9IiIpCiAgICAgICAgcmV0dXJuCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHRyeToKICAgICAgICBtYWluKCkKICAgIGV4Y2VwdCBLZXlib2FyZEludGVycnVwdDoKICAgICAgICBzeXMuZXhpdCgxMzApCg==',
    'requirements.txt': 'ZXhjZXB0aW9uZ3JvdXA9PTEuMi4wCmg1cHk9PTMuMTAuMAppbmljb25maWc9PTIuMC4wCm1waTRweT49NC4wCm51bXB5PT0xLjI2LjQKcGFja2FnaW5nPT0yMy4yCnBsdWdneT09MS40LjAKcHl0ZXN0PT04LjAuMQpweXRlc3QtbXBpPT0wLjYKcHl0ZXN0LXJlcG9ydGxvZz09MC40LjAKUHlZQU1MPT02LjAuMQp0b21saT09Mi4wLjEK',
}

for name, payload in embedded_files.items():
    Path(name).write_bytes(base64.b64decode(payload))

!pwd
!ls -la student_kernel.py student_local_test.py student_submit.py requirements.txt

/content
-rw-r--r-- 1 root root   189 May  9 18:56 requirements.txt
-rw-r--r-- 1 root root  3259 May  9 18:56 student_kernel.py
-rw-r--r-- 1 root root 11492 May  9 18:56 student_local_test.py
-rw-r--r-- 1 root root 14125 May  9 18:56 student_submit.py
